## Stochastic Gradient Descent

### Custom Dataset Creation


In [1]:
import numpy as np

np.random.seed(42)

n = 100000
d = 5

X = np.random.randn(n, d)
w_true = np.random.randn(d)

sigma = 0.1
noise = sigma * np.random.randn(n)
y = X @ w_true + noise

print("X shape:", X.shape)
print("y shape:", y.shape)
print("w_true shape:", w_true.shape)


X shape: (100000, 5)
y shape: (100000,)
w_true shape: (5,)


<div style="background-color:#f5f7fa; padding:18px; border-left:6px solid #4a90e2">

### Objective Function Definition

We work with a **finite-sum optimization problem**, which is the standard form in
machine learning and empirical risk minimization.

---

### Per-sample Loss

For a single data point $(x_i, y_i)$, we define the squared loss:

$$
f_i(w) = \frac{1}{2}\big(w^\top x_i - y_i\big)^2
$$

This measures how well the parameter vector $w$ fits **one sample**.

---

### Total Loss (Empirical Risk)

The full objective is the average loss over all samples:

$$
f(w) = \frac{1}{n}\sum_{i=1}^{n} f_i(w)
$$

This is the function we ultimately want to **minimize**.

</div>


### Required Functions for Line Search

In [2]:
full_loss_calls = 0

def loss_full(w):
    r = X @ w - y
    return 0.5 * np.mean(r**2)

def grad_full(w):
    return (X.T @ (X @ w - y)) / n

def armijo_line_search(w, direction, grad, c=1e-4, beta=0.5, max_iter=100):
    global full_loss_calls

    eta = 1.0
    f_w = loss_full(w)
    full_loss_calls += 1

    for _ in range(max_iter):
        f_new = loss_full(w + eta * direction)
        full_loss_calls += 1

        if f_new <= f_w + c * eta * grad.dot(direction):
            return eta

        eta *= beta

    return eta


In [3]:
w = np.zeros(d)

full_loss_calls = 0
losses_line_search = []

max_iters = 1000

for k in range(max_iters):
    g = grad_full(w)
    direction = -g

    eta = armijo_line_search(w, direction, g)
    w = w + eta * direction

    current_loss = loss_full(w)
    losses_line_search.append(current_loss)

    print(
        f"Iter {k:02d} | eta = {eta:.4f} | "
        f"loss = {current_loss:.6f}"
    )

print("\n=== COST REPORT (LINE SEARCH + FULL GD) ===")
print("Total full loss evaluations:", full_loss_calls)
print("Total data points touched:", full_loss_calls * n)


Iter 00 | eta = 1.0000 | loss = 0.005039
Iter 01 | eta = 1.0000 | loss = 0.005023
Iter 02 | eta = 1.0000 | loss = 0.005023
Iter 03 | eta = 1.0000 | loss = 0.005023
Iter 04 | eta = 1.0000 | loss = 0.005023
Iter 05 | eta = 0.2500 | loss = 0.005023
Iter 06 | eta = 1.0000 | loss = 0.005023
Iter 07 | eta = 1.0000 | loss = 0.005023
Iter 08 | eta = 0.5000 | loss = 0.005023
Iter 09 | eta = 0.5000 | loss = 0.005023
Iter 10 | eta = 0.2500 | loss = 0.005023
Iter 11 | eta = 1.0000 | loss = 0.005023
Iter 12 | eta = 1.0000 | loss = 0.005023
Iter 13 | eta = 1.0000 | loss = 0.005023
Iter 14 | eta = 1.0000 | loss = 0.005023
Iter 15 | eta = 1.0000 | loss = 0.005023
Iter 16 | eta = 1.0000 | loss = 0.005023
Iter 17 | eta = 1.0000 | loss = 0.005023
Iter 18 | eta = 1.0000 | loss = 0.005023
Iter 19 | eta = 1.0000 | loss = 0.005023
Iter 20 | eta = 1.0000 | loss = 0.005023
Iter 21 | eta = 1.0000 | loss = 0.005023
Iter 22 | eta = 1.0000 | loss = 0.005023
Iter 23 | eta = 1.0000 | loss = 0.005023
Iter 24 | eta = 

### Required Functions for SGD

In [4]:
stochastic_grad_calls = 0
full_loss_calls_SGD = 0

def grad_stochastic(w, i):
    global stochastic_grad_calls
    stochastic_grad_calls += 1
    return (w @ X[i] - y[i]) * X[i]

def loss_full_SGD(w):
    global full_loss_calls_SGD
    full_loss_calls_SGD += 1
    r = X @ w - y
    return 0.5 * np.mean(r**2)



In [5]:
w = np.zeros(d)

stochastic_grad_calls = 0
full_loss_calls_SGD = 0

losses_sgd = []

eta = 0.01
max_iters = 1000

for k in range(max_iters):
    i = np.random.randint(n)
    g = grad_stochastic(w, i)
    w = w - eta * g

    # store loss for plotting (same as line search)
    current_loss = loss_full_SGD(w)
    losses_sgd.append(current_loss)

    if k % 2 == 0:
        print(f"Iter {k:02d} | loss = {current_loss:.6f}")

print("\n=== COST REPORT (PURE SGD) ===")
print("Stochastic gradient evaluations:", stochastic_grad_calls)
print("Full loss evaluations (logging only):", full_loss_calls_SGD)
print("Total data points touched (SGD):", stochastic_grad_calls)


Iter 00 | loss = 1.251112
Iter 02 | loss = 1.237418
Iter 04 | loss = 1.232429
Iter 06 | loss = 1.128117
Iter 08 | loss = 1.043770
Iter 10 | loss = 1.022946
Iter 12 | loss = 0.988868
Iter 14 | loss = 0.909649
Iter 16 | loss = 0.903106
Iter 18 | loss = 0.878354
Iter 20 | loss = 0.858709
Iter 22 | loss = 0.837072
Iter 24 | loss = 0.666954
Iter 26 | loss = 0.632600
Iter 28 | loss = 0.628284
Iter 30 | loss = 0.605047
Iter 32 | loss = 0.562738
Iter 34 | loss = 0.553887
Iter 36 | loss = 0.553648
Iter 38 | loss = 0.530312
Iter 40 | loss = 0.527511
Iter 42 | loss = 0.520272
Iter 44 | loss = 0.509367
Iter 46 | loss = 0.500979
Iter 48 | loss = 0.487909
Iter 50 | loss = 0.469441
Iter 52 | loss = 0.449181
Iter 54 | loss = 0.446917
Iter 56 | loss = 0.444710
Iter 58 | loss = 0.444869
Iter 60 | loss = 0.422188
Iter 62 | loss = 0.412547
Iter 64 | loss = 0.407066
Iter 66 | loss = 0.373350
Iter 68 | loss = 0.329364
Iter 70 | loss = 0.298726
Iter 72 | loss = 0.296296
Iter 74 | loss = 0.296010
Iter 76 | lo

In [6]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=losses_line_search,
        mode="lines+markers",
        name="Line Search (Full GD)",
        line=dict(width=3)
    )
)

fig.add_trace(
    go.Scatter(
        y=losses_sgd,
        mode="lines+markers",
        name="Pure SGD",
        line=dict(width=3, dash="dash")
    )
)

fig.update_layout(
    title="Loss Curves: Line Search vs Pure SGD",
    xaxis_title="Iteration",
    yaxis_title="Loss",
    template="plotly_white",
    width=900,
    height=500,
    legend=dict(x=0.02, y=0.98)
)

fig.show()


In [7]:
# x-axis = cumulative data touched
x_line_search = np.arange(1, len(losses_line_search) + 1) * n
x_sgd = np.arange(1, len(losses_sgd) + 1)


In [8]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x_line_search,
        y=losses_line_search,
        mode="lines+markers",
        name="Line Search (Full GD)",
        line=dict(width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=x_sgd,
        y=losses_sgd,
        mode="lines+markers",
        name="Pure SGD",
        line=dict(width=3)
    )
)

fig.update_layout(
    title="Loss vs Data Touched: Line Search vs SGD",
    xaxis_title="Total Data Points Touched",
    yaxis_title="Loss",
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)

fig.show()



# Line Search vs Stochastic Gradient Descent  
### Empirical Comparison: Solution Quality vs Data Efficiency


## Experimental Setup

| Component | Value |
|---------|------|
| Objective | Mean Squared Error |
| Dataset size (n) | 100,000 |
| Parameter dimension (d) | Same for both methods |
| Initialization | w = 0 |
| Iterations | 1000 |
| Evaluation | Full loss on entire dataset |

The objective function and data are identical.  
Only the **optimization strategy** differs.



## Line Search with Full Gradient Descent

### Observed Result

| Metric | Value |
|------|------|
| Final loss | 0.005023 |
| Iterations | 1000 |

### Computational Cost (Measured)

| Quantity | Value |
|--------|------|
| Full loss evaluations | 2006 |
| Data points per evaluation | 100,000 |
| **Total data points touched** | **200,600,000** |

### Interpretation

- Each iteration computes a **full gradient**
- Line search repeatedly evaluates the **entire loss surface**
- Near convergence, most computation is spent **verifying step sizes**, not reducing loss

---


## Pure Stochastic Gradient Descent (SGD)

### Observed Result

| Metric | Value |
|------|------|
| Final loss | 0.00507 |
| Iterations | 1000 |

### Computational Cost (Measured)

| Quantity | Value |
|--------|------|
| Stochastic gradient evaluations | 1000 |
| Data points per update | 1 |
| **Total data points touched** | **1000** |

### Interpretation

- Each iteration uses **one data point**
- Loss decreases noisily but consistently
- No repeated scans of the full dataset



## Direct Comparison

| Aspect | Line Search + GD | SGD |
|------|-----------------|-----|
| Objective minimized | Same | Same |
| Final solution quality | High | High |
| Per-iteration cost | Very high | Very low |
| Data efficiency | Poor | Extremely high |
| Scalability to large n | No | Yes |



## Key Conclusion

Both methods reach nearly identical solutions.  
The decisive difference is **how much data is accessed to get there**.

> Line search optimizes **step correctness**,  
> SGD optimizes **data efficiency**.




## Line Search vs SGD — Behaviour Near the Optimum


Both methods optimize the **same objective** on the **same dataset**.

However, the final loss values are different:

| Method | Final Loss |
|------|------------|
| Line Search + Full Gradient Descent | **0.005023** |
| Pure SGD (constant step size) | **≈ 0.00507** |

This difference does **not** come from:
- model capacity
- data quality
- objective definition

It comes from **how the updates behave near the optimum**.

To understand why:

- Line Search settles at **0.005023**
- SGD stalls around **0.00507**

we zoom in on the **loss trajectory near the minimum**.



In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=losses_line_search,
        mode="lines",
        name="Line Search (Full GD)",
        line=dict(width=3)
    )
)

fig.add_trace(
    go.Scatter(
        y=losses_sgd,
        mode="lines",
        name="Pure SGD",
        line=dict(width=3)
    )
)

fig.update_layout(
    title="Loss Near Optimum: Line Search vs SGD (Oscillation View)",
    xaxis_title="Iteration",
    yaxis_title="Loss",
    height=450,
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)

fig.update_yaxes(range=[0.0048, 0.0053])

fig.show()



## Loss Near the Optimum — What This Plot Shows


From the zoomed-in view near convergence:

| Method | Behaviour Near Optimum |
|------|------------------------|
| **Line Search (Full GD)** | Smooth, flat convergence at **0.005023** |
| **Pure SGD** | Persistent oscillations around **≈ 0.00507** |

Both methods reach the **same basin**,  
but only one **settles**.

---

## Why This Happens

### Line Search (Full Gradient)
- Uses **exact gradient of the full objective**
- Adapts step size to satisfy **descent conditions**
- Near optimum:
  - step size shrinks automatically
  - updates align with local curvature
- Result: **stable convergence**


### Pure SGD (Constant Step Size)
- Uses **noisy gradient from a single data point**
- Fixed learning rate (`η = 0.01`)
- Near optimum:
  - gradient noise does **not vanish**
  - constant step keeps pushing parameters back and forth
- Result: **oscillation instead of settling**

This is **not divergence** —  
it is a **noise floor**.

---


## Key Insight

> SGD does **not converge to a point** with constant step size.  
> It converges to a **neighbourhood** of the optimum.

The radius of that neighbourhood is controlled by:
- gradient noise
- learning rate



## How This Is Fixed in Practice

| Technique | Effect |
|---------|--------|
| Learning rate decay | Shrinks oscillations |
| Mini-batch SGD | Reduces gradient variance |
| Averaging (Polyak / EMA) | Smooths final solution |
| Adaptive optimizers | Implicit step-size control |

Modern optimizers exist **because of this exact plot**.




<div style="text-align:center; margin-top:40px; margin-bottom:40px;">

<span style="font-size:36px; font-weight:700; color:#2563EB;">
Mini-Batch Stochastic Gradient Descent
</span>

<br>

<span style="font-size:18px; color:#475569;">
The bridge between noisy SGD and expensive Full Gradient Descent
</span>

</div>


In [14]:
def run_minibatch_sgd(batch_size, max_iters=1000, eta=0.01):
    w = np.zeros(d)
    losses = []

    for k in range(max_iters):
        idx = np.random.choice(n, batch_size, replace=False)
        g = np.zeros(d)

        for i in idx:
            g += (w @ X[i] - y[i]) * X[i]

        g /= batch_size
        w = w - eta * g

        # store loss every iteration (IMPORTANT)
        r = X @ w - y
        loss = 0.5 * np.mean(r**2)
        losses.append(loss)

    return losses


In [15]:
losses_b8   = run_minibatch_sgd(batch_size=8)
losses_b16  = run_minibatch_sgd(batch_size=16)
losses_b32  = run_minibatch_sgd(batch_size=32)
losses_b64  = run_minibatch_sgd(batch_size=64)
losses_b128 = run_minibatch_sgd(batch_size=128)


In [17]:
print("=== MINI-BATCH SGD: COST REPORT ===\n")
print("Batch Size |   Final Loss |  Data Points Touched")
print("------------------------------------------------")

n_iters = len(losses_b8)

batch_loss_map = {
    8:   losses_b8,
    16:  losses_b16,
    32:  losses_b32,
    64:  losses_b64,
    128: losses_b128,
}

for batch_size, losses in batch_loss_map.items():
    final_loss = losses[-1]
    data_touched = batch_size * len(losses)

    print(f"{batch_size:10d} | {final_loss:10.6f} | {data_touched:20d}")


=== MINI-BATCH SGD: COST REPORT ===

Batch Size |   Final Loss |  Data Points Touched
------------------------------------------------
         8 |   0.005046 |                 8000
        16 |   0.005032 |                16000
        32 |   0.005024 |                32000
        64 |   0.005023 |                64000
       128 |   0.005026 |               128000


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "Batch = 8", "Batch = 16",
        "Batch = 32", "Batch = 64",
        "Batch = 128", ""
    )
)

# Row 1
fig.add_trace(go.Scatter(y=losses_b8,  mode="lines", name="Batch = 8"),  row=1, col=1)
fig.add_trace(go.Scatter(y=losses_b16, mode="lines", name="Batch = 16"), row=1, col=2)

# Row 2
fig.add_trace(go.Scatter(y=losses_b32, mode="lines", name="Batch = 32"), row=2, col=1)
fig.add_trace(go.Scatter(y=losses_b64, mode="lines", name="Batch = 64"), row=2, col=2)

# Row 3
fig.add_trace(go.Scatter(y=losses_b128, mode="lines", name="Batch = 128"), row=3, col=1)

# Layout
fig.update_layout(
    title="Mini-Batch SGD: Loss Oscillations Near Optimum",
    template="plotly_white",
    height=900,
    showlegend=False
)

fig.update_yaxes(range=[0.00502, 0.00506])

fig.show()


In [ ]:
true_opt_loss = 0.005023

for r in range(1, 4):
    for c in range(1, 3):
        fig.add_hline(
            y=true_opt_loss,
            line=dict(color="black", width=2, dash="dash"),
            row=r,
            col=c
        )

fig.show()


### Final Conclusion

<div align="center">

<table>
<thead>
<tr style="background-color:#1f2937;color:white;">
<th>Method</th>
<th>Accuracy</th>
<th>Cost per Iteration</th>
<th>Noise</th>
</tr>
</thead>
<tbody>
<tr style="background-color:#f3f4f6;">
<td><b>Full GD</b></td>
<td>Exact</td>
<td>Very High</td>
<td>None</td>
</tr>
<tr style="background-color:#e5e7eb;">
<td><b>SGD</b></td>
<td>Approx</td>
<td>Very Low</td>
<td>High</td>
</tr>
<tr style="background-color:#f3f4f6;">
<td><b>Mini-Batch SGD</b></td>
<td>Near</td>
<td>Medium</td>
<td>Medium</td>
</tr>
</tbody>
</table>

</div>
